In [1]:
# etl-bronze.py

import requests
import os
from datetime import datetime
from pyspark.sql import SparkSession

# Inicializa sessão Spark
spark = SparkSession.builder \
    .appName("ETL Bronze IBGE") \
    .getOrCreate()

# Lê variáveis de ambiente
storage_account = os.getenv("AZURE_STORAGE_ACCOUNT")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

# Configuração de autenticação Azure (Service Principal)
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# URLs IBGE
IBGE_BR_ESTADOS = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
IBGE_BR_MUNICIPIOS = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

# Função para consumir API e transformar em DataFrame Spark
def api_to_df(url, spark):
    response = requests.get(url)
    data = response.json()
    rdd = spark.sparkContext.parallelize(data)
    df = spark.read.json(rdd)
    return df

# Extract
df_estados = api_to_df(IBGE_BR_ESTADOS, spark)
df_municipios = api_to_df(IBGE_BR_MUNICIPIOS, spark)

# Transform (mínimo, pois Bronze guarda dados crus)
df_estados_bronze = df_estados.select("id", "nome", "sigla", "regiao.nome")
df_municipios_bronze = df_municipios.select("id", "nome", "microrregiao.mesorregiao.UF.sigla")

# Prefixo de data AAAA-MM-DD
prefixo_data = datetime.today().strftime("%Y-%m-%d")

# Caminhos na Azure Data Lake (camada Bronze)
path_estados = f"abfss://bronze@{storage_account}.dfs.core.windows.net/ibge_estados/{prefixo_data}"
path_municipios = f"abfss://bronze@{storage_account}.dfs.core.windows.net/ibge_municipios/{prefixo_data}"

# Load (grava em Parquet na camada Bronze)
df_estados_bronze.write.mode("overwrite").parquet(path_estados)
df_municipios_bronze.write.mode("overwrite").parquet(path_municipios)

print("Ingestão concluída com sucesso na camada Bronze!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/22 22:12:35 WARN Utils: Your hostname, DESKTOP-PFG30PT, resolves to a loopback address: 127.0.1.1; using 172.23.114.15 instead (on interface eth0)
26/08/22 22:12:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/22 22:12:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Py4JJavaError: An error occurred while calling o71.parquet.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.azurebfs.SecureAzureBlobFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2724)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:135)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWritingFileFormat(DataSource.scala:485)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWriting(DataSource.scala:572)
	at org.apache.spar